In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from src.retrieval.retriever import Retriever
from src.retrieval.reranker import Reranker
from src.evaluation.retrieval_metrics import load_eval_questions, recall_at_k, mrr

retriever = Retriever(persist_dir='../vector_store', collection_name='sr_papers')
reranker  = Reranker()
questions = load_eval_questions('../data/processed/eval_questions.csv')
answerable = [q for q in questions if q['difficulty'] != 'unanswerable']
print(f'Ready. {len(answerable)} answerable questions.')

In [ ]:
query = "What loss function does SRGAN use?"

# Dense only
dense = retriever.retrieve(query, top_k=5)['results']

# Dense + rerank
candidates = retriever.retrieve(query, top_k=15)['results']
reranked   = reranker.rerank(query, candidates, top_k=5)

print(f"Query: {query}\n")
print("=== Dense retrieval ===")
for r in dense:
    print(f"  [{r['citation_index']}] {r['method']} | p{r['page_number']} | score={r['score']:.3f}")

print("\n=== After reranking ===")
for r in reranked:
    print(f"  [{r['citation_index']}] {r['method']} | p{r['page_number']} | dense={r['dense_score']:.3f} | reranker={r['reranker_score']:.3f}")

In [ ]:
import matplotlib.pyplot as plt

query = "What loss function does SRGAN use?"
candidates = retriever.retrieve(query, top_k=15)['results']
reranked   = reranker.rerank(query, candidates, top_k=5)

labels = [f"{r['method']}\np{r['page_number']}" for r in reranked]
dense_scores    = [r['dense_score']    for r in reranked]
reranker_scores = [r['reranker_score'] for r in reranked]

# Normalize reranker scores to [0,1] for visual comparison
min_r, max_r = min(reranker_scores), max(reranker_scores)
norm_reranker = [(s - min_r) / (max_r - min_r + 1e-8) for s in reranker_scores]

x = range(len(labels))
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar([i - 0.2 for i in x], dense_scores,    0.4, label='Dense score',            color='steelblue')
ax.bar([i + 0.2 for i in x], norm_reranker,   0.4, label='Reranker score (normed)', color='darkorange')
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel('Score')
ax.set_title(f'Dense vs Reranker scores\n"{query}"')
ax.legend()
plt.tight_layout()
plt.savefig('../docs/figures/reranker_comparison.png', dpi=150)
plt.show()

In [ ]:
import time

rows = []
for q in answerable:
    # Dense only
    t0 = time.time()
    dense_results = retriever.retrieve(q['question'], top_k=5)['results']
    dense_ms = (time.time() - t0) * 1000
    dense_methods = [r['method'] for r in dense_results]

    # Dense + rerank
    t0 = time.time()
    candidates = retriever.retrieve(q['question'], top_k=15)['results']
    reranked_results = reranker.rerank(q['question'], candidates, top_k=5)
    rerank_ms = (time.time() - t0) * 1000
    reranked_methods = [r['method'] for r in reranked_results]

    em = q['expected_methods']
    rows.append({
        'Q':              q['question_id'],
        'Difficulty':     q['difficulty'],
        'Dense R@5':      round(recall_at_k(dense_methods,    em, k=5), 2),
        'Reranked R@5':   round(recall_at_k(reranked_methods, em, k=5), 2),
        'Dense MRR':      round(mrr(dense_methods,    em), 2),
        'Reranked MRR':   round(mrr(reranked_methods, em), 2),
        'Dense ms':       round(dense_ms,  0),
        'Rerank ms':      round(rerank_ms, 0),
    })

df = pd.DataFrame(rows)
df['R@5 delta'] = df['Reranked R@5'] - df['Dense R@5']
df['MRR delta'] = df['Reranked MRR'] - df['Dense MRR']
df

In [ ]:
print("=== Aggregate comparison ===\n")
print(f"Dense    Recall@5 : {df['Dense R@5'].mean():.3f}")
print(f"Reranked Recall@5 : {df['Reranked R@5'].mean():.3f}")
print(f"Delta              : {df['R@5 delta'].mean():+.3f}")
print()
print(f"Dense    MRR      : {df['Dense MRR'].mean():.3f}")
print(f"Reranked MRR      : {df['Reranked MRR'].mean():.3f}")
print(f"Delta             : {df['MRR delta'].mean():+.3f}")
print()
print(f"Dense avg latency  : {df['Dense ms'].mean():.0f} ms")
print(f"Rerank avg latency : {df['Rerank ms'].mean():.0f} ms")
print(f"Extra latency      : +{df['Rerank ms'].mean() - df['Dense ms'].mean():.0f} ms")

In [ ]:
delta_recall = df['Reranked R@5'].mean() - df['Dense R@5'].mean()
delta_latency = df['Rerank ms'].mean() - df['Dense ms'].mean()

print("=== Decision ===\n")
if delta_recall > 0.05:
    print(f"Reranker improves Recall@5 by {delta_recall:+.3f} — worth the {delta_latency:.0f}ms latency cost.")
    print("Recommendation: enable reranker by default.")
elif delta_recall > 0:
    print(f"Reranker gives a small gain ({delta_recall:+.3f}). Consider enabling for quality-sensitive queries only.")
else:
    print(f"Reranker shows no gain on this corpus. Dense retrieval is sufficient.")

print()
print("Future improvement: Hybrid search (BM25 + dense)")
print("  Dense retrieval can miss exact technical terms like 'RCAN' or")
print("  'perceptual loss' when they are rare in the embedding space.")
print("  BM25 keyword matching combined with dense retrieval (hybrid search)")
print("  is the standard fix — add this to the README as a planned improvement.")